In [9]:
import sys
print(sys.executable)

/Users/jillianbaker/Desktop/STAT 386/Final-Project-386/venv/bin/python


In [1]:
# Imports 

import pandas as pd
from data_collection import get_options_chain

In [2]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]

KEEP_COLS = [
    "ticker",
    "expiration",
    "contractType",   # renamed from option_type for clarity
    "strike",
    "bid",
    "ask",
    "volume",
    "openInterest",
    "impliedVolatility",
]

OUTPUT_CSV     = "options_chain.csv"
OUTPUT_PARQUET = "options_chain.parquet"

In [3]:
# Fetch options chain for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching options chain: {ticker} ...", end=" ")
    try:
        df = get_options_chain(ticker)

        # Rename option_type -> contractType to match acceptance criteria
        df = df.rename(columns={"option_type": "contractType"})

        # Keep only the required columns (drop any that don't exist gracefully)
        cols = [c for c in KEEP_COLS if c in df.columns]
        df = df[cols]

        frames.append(df)
        print(f"✓  {len(df):,} rows")

    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

options = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows collected: {len(options):,}")

Fetching options chain: VOO ... 

INFO | get_options_chain('VOO'): 1654 rows across 16 expirations.


✓  1,654 rows
Fetching options chain: QQQ ... 

INFO | get_options_chain('QQQ'): 7170 rows across 30 expirations.


✓  7,170 rows
Fetching options chain: ARKQ ... 

INFO | get_options_chain('ARKQ'): 225 rows across 5 expirations.


✓  225 rows
Fetching options chain: BOTZ ... 

INFO | get_options_chain('BOTZ'): 96 rows across 4 expirations.


✓  96 rows

Total rows collected: 9,145


In [4]:
# Light cleaning

# Ensure correct dtypes
options["expiration"]       = pd.to_datetime(options["expiration"])
options["strike"]           = pd.to_numeric(options["strike"],           errors="coerce")
options["bid"]              = pd.to_numeric(options["bid"],              errors="coerce")
options["ask"]              = pd.to_numeric(options["ask"],              errors="coerce")
options["volume"]           = pd.to_numeric(options["volume"],           errors="coerce")
options["openInterest"]     = pd.to_numeric(options["openInterest"],     errors="coerce")
options["impliedVolatility"]= pd.to_numeric(options["impliedVolatility"],errors="coerce")

# Drop rows where strike or IV is missing — not useful for analysis
before = len(options)
options = options.dropna(subset=["strike", "impliedVolatility"])
dropped = before - len(options)
if dropped:
    print(f"Dropped {dropped:,} rows with missing strike or impliedVolatility.")

# Sort for readability
options = options.sort_values(
    ["ticker", "expiration", "contractType", "strike"]
).reset_index(drop=True)

options.head(10)


,ticker,expiration,contractType,strike,bid,ask,volume,openInterest,impliedVolatility
0,ARKQ,2026-04-17,call,105.0,7.30,9.70,16.0,1,0.549321
1,ARKQ,2026-04-17,call,110.0,3.60,5.50,12.0,6,0.434576
2,ARKQ,2026-04-17,call,115.0,1.65,2.65,5.0,3,0.387091
3,ARKQ,2026-04-17,call,116.0,1.35,2.40,6.0,8,0.398932
4,ARKQ,2026-04-17,call,117.0,0.90,2.05,5.0,5,0.396124
5,ARKQ,2026-04-17,call,118.0,0.75,1.70,15.0,18,0.388922
6,ARKQ,2026-04-17,call,119.0,0.20,1.90,1.0,2,0.444341
7,ARKQ,2026-04-17,call,120.0,0.30,1.20,3.0,12,0.385504
8,ARKQ,2026-04-17,call,121.0,0.10,1.05,1.0,5,0.391608
9,ARKQ,2026-04-17,call,122.0,0.05,0.80,2.0,6,0.378424


In [5]:
# Quick summary

summary = (
    options.groupby(["ticker", "contractType"])
    .agg(
        expirations   = ("expiration", "nunique"),
        contracts     = ("strike",     "count"),
        avg_iv        = ("impliedVolatility", "mean"),
        total_oi      = ("openInterest", "sum"),
    )
    .round(4)
)
print(summary)

                     expirations  contracts  avg_iv  total_oi
ticker contractType                                          
ARKQ   call                    5        135  0.4992      1957
       put                     5         90  0.3300      1490
BOTZ   call                    4         58  0.4989      4771
       put                     4         38  0.5088      6001
QQQ    call                   30       3773  0.3505   3400227
       put                    30       3397  0.3175   4727535
VOO    call                   16        841  0.3444     53899
       put                    16        813  0.3455     38500


In [8]:
# Save to disk

options.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV     → {OUTPUT_CSV}  ({options.shape[0]:,} rows)")

Saved CSV     → options_chain.csv  (9,145 rows)
